In [ ]:
import great_expectations as gx

context = gx.get_context()

# Ustvari nov podatkovni vir
datasource_name = "nesrece"
datasource = context.sources.add_pandas_filesystem(
    name=datasource_name,
    base_directory="/Users/jozelavric/Desktop/Sola/ISS/ISSProjekt/data"
)

# Dodaj podatkovni vir
data_asset_name = "nesrece_data"

data_asset = datasource.add_csv_asset(
    name=data_asset_name,
    batching_regex=r"preprocessed/nesrece_v_cestnem_prometu\.csv"
)

In [ ]:
expectation_suite_name = "nesrece_suite"
expectation_suite = context.add_or_update_expectation_suite(
    expectation_suite_name=expectation_suite_name
)

In [ ]:
asset = context.get_datasource(datasource_name).get_asset(data_asset_name)

batch_request = asset.build_batch_request()
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=expectation_suite_name
)

exclude_column_names = []
data_assistant_result = context.assistants.onboarding.run(
    validator=validator,
    exclude_column_names=exclude_column_names
)

expectation_suite = data_assistant_result.get_expectation_suite()
context.save_expectation_suite(
    expectation_suite=expectation_suite,
    expectation_suite_name=expectation_suite_name
)

In [ ]:
# prijavaCas ne sme biti NULL
validator.expect_column_values_to_not_be_null("prijavaCas")

# obcinaNaziv ne sme biti NULL in mora biti tipa string
validator.expect_column_values_to_not_be_null("obcinaNaziv")
validator.expect_column_values_to_be_of_type("obcinaNaziv", "str")

# nastanekCas ne sme biti NULL
validator.expect_column_values_to_not_be_null("nastanekCas")

# Vsi trije stolpci morajo biti prisotni
validator.expect_table_columns_to_match_ordered_list(
    ["prijavaCas", "obcinaNaziv", "nastanekCas"]
)

# Število vrstic mora biti > 0
validator.expect_table_row_count_to_be_between(min_value=1)

# Validacija ISO 8601 časovnega formata
validator.expect_column_values_to_match_regex(
    "prijavaCas",
    r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}"
)
validator.expect_column_values_to_match_regex(
    "nastanekCas",
    r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}"
)

validator.save_expectation_suite(discard_failed_expectations=False)

In [ ]:
checkpoint_name = "nesrece_checkpoint"
batch_request = asset.build_batch_request()

checkpoint = context.add_or_update_checkpoint(
    name=checkpoint_name,
    validations=[
        {
            "batch_request": batch_request,
            "expectation_suite_name": expectation_suite_name
        }
    ],
)

checkpoint_result = checkpoint.run()
context.build_data_docs()
context.open_data_docs()